# 4장 1강: 데이터 불균형과 평가지표 왜곡

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 데이터 불균형이 분류 모델 평가에 어떤 영향을 주는지 확인합니다.

- `Class` 비율을 확인하여 불균형 정도를 파악합니다.
- Accuracy만 보면 생길 수 있는 착시를 확인합니다.
- Precision, Recall, F1-score, PR-AUC를 함께 확인합니다.
- `class_weight`, Random Undersampling, SMOTE를 적용하여 성능 변화를 비교합니다.
- threshold 변화에 따른 Precision / Recall 변화를 확인합니다.

> 이 실습에서는 4장 1강 교안에 나온 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습에서는 모델 입력으로 `V1`부터 `V28`까지 사용하고, `Class`를 예측 대상으로 사용합니다.

## 실습 준비

1. 데이터 크기를 확인합니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 X, `Class`를 y로 분리합니다.
4. Train / Test 데이터를 분리합니다.
5. 리샘플링은 **Train 데이터에만** 적용합니다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

data = pd.read_csv("creditcard.csv")

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(data["Class"].value_counts().to_frame("count"))

print("\n[Class 비율]")
display(data["Class"].value_counts(normalize=True).to_frame("ratio"))

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727



Train 크기: (227845, 28)
Test 크기: (56962, 28)


---

# 필수 1. Accuracy의 함정과 불균형 지표 확인

## 배경

Credit Card Fraud 데이터는 정상 거래가 매우 많고 사기 거래가 매우 적은 불균형 데이터입니다.

이런 상황에서는 Accuracy만 보면 모델이 매우 좋아 보일 수 있습니다.

## 문제 1. 모든 거래를 정상이라고 예측해보세요.

### 요구사항

1. Test 데이터의 모든 거래를 `0`으로 예측합니다.
2. Accuracy를 계산합니다.
3. Recall을 계산합니다.
4. Accuracy와 Recall 결과를 비교합니다.

### 확인 포인트

- 모든 거래를 정상이라고 예측해도 Accuracy가 매우 높게 나오는가?
- 사기 거래를 하나도 잡지 못하면 Recall은 어떻게 되는가?

In [2]:
# TODO
# 모든 거래를 정상(0)으로 예측하고 Accuracy / Recall을 계산하세요.


# Test 데이터의 모든 거래를 0으로 예측
y_pred = np.zeros(len(y_test), dtype=int)

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Accuracy:", round(accuracy, 4))
print("Recall:", round(recall, 4))


Accuracy: 0.9983
Recall: 0.0


- 결과   

|실제 정답|정상 0으로 예측|사기 1로 예측|내용|
|---|---|---|---|
|정상 0|`TN` = 56,864|`FP` = 0|정상 거래는 전부 맞춤|
|사기 1|`FN` = 98|`TP` = 0|사기 거래는 전부 다 틀림|

- -> `TN`은 정상을 정상으로 맞춘 건수
- -> `TP`는 사기를 사기로 맞춘 건수
- -> `FP`는 정상인데 사기라고 잘못 예측한 오탐
- -> `FN`은 사기인데 정상이라고 잘못 예측한 미탐

---

### Q1. Accuracy가 높더라도 이 모델을 좋은 사기 탐지 모델이라고 보기 어려운 이유는 무엇인가요?

**답변:**  
- 데이터의 대부분이 정상거래이기 때문에 전부 다 정상거래로 예측을 해도 정확도(Accuracy)는 높음
- 하지만 사기를 탐지해야하는데 Accuracy 하나만으로는 이 모델의 성능을 평가할 수 없음
- Recall, precision 등도 같이 확인해야 모델의 성능을 평가할 수 있음

- 불균형 데이터에서는 Accuracy가 높아도 소수 클래스를 전혀 탐지 못할 수 있다.

## 문제 2. 기본 Logistic Regression의 여러 지표를 확인하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 학습합니다.
2. Test 데이터를 예측합니다.
3. Accuracy를 계산합니다.
4. Precision을 계산합니다.
5. Recall을 계산합니다.
6. F1-score를 계산합니다.
7. `predict_proba()`의 양성 클래스 확률로 PR-AUC를 계산합니다.
8. 혼동행렬을 확인합니다.

### 확인 포인트

- Accuracy 하나만 보는 것과 여러 지표를 함께 보는 것의 차이를 이해했는가?
- Precision과 Recall이 각각 어떤 오류와 연결되는지 설명할 수 있는가?
- PR-AUC가 불균형 데이터에서 왜 중요한지 이해했는가?

In [3]:
# TODO
# 기본 Logistic Regression의 Accuracy / Precision / Recall / F1 / PR-AUC / 혼동행렬을 확인하세요.

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)


# 예측값
y_pred = lr_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print(f'Accuracy : {accuracy:.3f}')
print(f'Precision : {precision:.3f}')
print(f'Recall : {recall:.3f}')
print(f'F1-score : {f1:.3f}')

# 예측확률
y_pred_proba = lr_model.predict_proba(X_test)[:, 1] # 사기거래 1에 대한 데이터만 추출
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f'PR-AUC : {pr_auc:.4f}')

# 혼동행렬
matrix = confusion_matrix(y_test, y_pred)

print('\n[Confusion Matrix]')
print(matrix)

Accuracy : 0.999
Precision : 0.829
Recall : 0.643
F1-score : 0.724
PR-AUC : 0.7436

[Confusion Matrix]
[[56851    13]
 [   35    63]]


- 혼동행렬   

|실제 정답/ 예측|정상0|사기|실제 거래 합계|
|---|---|---|---|
|정상 0|`TN` = 56,851|`FP` = 13|56,864|
|사기 1|`FN` = 35|`TP` = 63|98|
|예측 합계|56,886|76|56,962|

- `TP` : 사기 사건 63건을 사기라고 올바르게 탐지한 경우
- `FN` : 사기 35건을 정상이라고 잘못 예측한 경우
- `FP` : 정상 13건을 사기라고 잘못 예측한 경우
- `TN` : 정상 56,851건을 정상이라고 올바르게 예측한 경우

    - -> 사기라고 예측한 거래는 TP + FP = 76건
    - -> 실제 사기 거래는 TP + FN = 98건

|지표|계산 식|결과|내용|
|---|---|---|---|
|Accuracy|(56,851 + 63) / 56,962|0.999|전체 거래의 약 99.9%를 올바르게 분류| 
|Precision|63 / (63 + 13)|0.828|사기라고 예측한 76건 둥 약 82.8%가 실제 사기| 
|Recall|63 / (63 + 35)|0.724|실제 사기 95건 중 64.2% 정도를 올바르게 분류| 
|F1-Score|2 * 63 / (2 * 63 + 13 + 35)|0.724|Precision과 Recall의 조화 평균| 
|PR-AUC|정답과 사기 점수|0.743|Precision-Recall 성능요약| 

<br>

- Precision은 사기라고 말한 것을 얼마나 믿을 수 있는지?
- Recall은 사기를 실제로 얼마나 놓지지 않고 찾았는지

> **해석 :**   
> 모두 정상으로 예측한 기준선은 0건을 탐지해냄, 기본 모델의 경우 63건을 찾고 35건을 놓쳤으며,   
> 정상 13건을 사기로 잘못 예측함   
> 사기를 사기로 탐지하는 능력은 있고 정확도도 매우 높은 편이지만     
> Recall이 64%정도이기 때문에 사기를 거의 다 잡아낼 수 있는 모델이다라고 평가할 수 없다.


### Q2. 사기 거래를 놓치는 FN의 비용이 크다면 Precision과 Recall 중 무엇을 더 중요하게 봐야 하나요?

**답변:**  
- **Recall**
- 실제 사기 중에서 얼마나 사기를 정확하게 찾아냈는지를 나타내기 때문
- 사기를 놓치는 FN을 중요하게 생각한다면 Recall값을 최대한 높여서 만들어야함

---

# 필수 2. 불균형 처리 기법 비교

## 배경

교안에서는 불균형 처리 방법으로 다음 세 가지를 다룹니다.

- `class_weight`
- Random Undersampling
- SMOTE

이번 문제에서는 세 방법을 Train 데이터에만 적용한 뒤 같은 Test 데이터에서 성능을 비교합니다.

## 문제 1. class_weight를 적용하세요.

### 요구사항

1. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
2. Train 데이터로 학습합니다.
3. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스에 더 큰 가중치를 주면 Recall이 어떻게 변하는가?
- Precision과 Recall이 동시에 항상 좋아지는 것은 아니라는 점을 확인했는가?

In [4]:
# TODO
# class_weight="balanced" 모델을 학습하고 지표를 확인하세요.

weighted = LogisticRegression(class_weight='balanced', max_iter=1000)

weighted.fit(X_train, y_train)

# 예측값
y_pred = weighted.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print(f'Accuracy : {accuracy:.3f}')
print(f'Precision : {precision:.3f}')
print(f'Recall : {recall:.3f}')
print(f'F1-score : {f1:.3f}')

# 예측확률
y_pred_proba = weighted.predict_proba(X_test)[:, 1] # 사기거래 1에 대한 데이터만 추출
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f'PR-AUC : {pr_auc:.4f}')

Accuracy : 0.974
Precision : 0.058
Recall : 0.918
F1-score : 0.109
PR-AUC : 0.7092


- balanced는 어떤 가중치를 부여하는지?
- 클래스 K의 학습표본 수가 NK 전체학습 표본 수가 N 클래스 수가 K면

```math
w_k \frac{n}{K n_K} 
```

- 가중치가 0.50정도 사기 한건이 약 289.14정도의 수치
- 같은 정도의 손실이라면 사기 한건이 정상 한건보다 약 577.3배 큰 비중을 가진다.

- `Precision` : 처음에 82% -> 지금 5% - 급격하게 사기라고 예측한 것 중에 실제 사기였던 비율이 크게 낮아짐
- `Recall` : 처음에 64% -> 지금 91% - 실제 사기를 기존보다 더 잘 예측
- `F1` : 처음에 72% -> 지금 10% - Precision의 값이 매우 작아 종합값이 매우 떨어짐

<br>

- `TP` : 기존 모델 63건, 클래스 가중치 부여한 경우 90건 -> 사기 사건 27건 추가 탐지
- `FN` : 기존 모델 35건, 클래스 가중치 부여한 경우 8건 -> 사기라고 탐지 못한 경우 27건 감소
- `FP` : 기존 모델 13건, 클래스 가중치 부여한 경우 1,465건 -> 오탐 1,452건 증가
- `TP + FP` : 기존 모델 76건, 클래스 가중치 부여한 경우 1,555건 -> 대부분 사기라고 예측을 해버림

---

## 문제 2. Random Undersampling을 적용하세요.

### 요구사항

1. `RandomUnderSampler(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 다수 클래스 샘플이 줄어드는가?
- Test 데이터에는 Undersampling을 적용하지 않았는가?

In [5]:
# TODO
# Random Undersampling을 Train 데이터에만 적용하세요.

under = RandomUnderSampler(random_state=42)

X_under, y_under = under.fit_resample(X_train, y_train)

under_model = LogisticRegression(max_iter=1000, random_state=42)

under_model.fit(X_under, y_under)

# 예측값
y_pred = under_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print(f'Accuracy : {accuracy:.3f}')
print(f'Precision : {precision:.3f}')
print(f'Recall : {recall:.3f}')
print(f'F1-score : {f1:.3f}')

# 예측확률
y_pred_proba = under_model.predict_proba(X_test)[:, 1] # 사기거래 1에 대한 데이터만 추출
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f'PR-AUC : {pr_auc:.4f}')

# 출력
print('\n[ Random UnderSampling 전 / 후 비교 ]')
print(f'UnderSampling 전 : {X_train.shape}')
print(f'UnderSampling 후 : {X_under.shape}')

Accuracy : 0.959
Precision : 0.038
Recall : 0.918
F1-score : 0.072
PR-AUC : 0.6358

[ Random UnderSampling 전 / 후 비교 ]
UnderSampling 전 : (227845, 28)
UnderSampling 후 : (788, 28)


## 문제 3. SMOTE를 적용하세요.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스 샘플이 증가하는가?
- SMOTE가 Test 데이터에 적용되지 않았는가?
- Recall 개선과 Precision 감소가 함께 나타날 수 있음을 확인했는가?

In [6]:
# TODO
# SMOTE를 Train 데이터에만 적용하세요.

smote = SMOTE(random_state=42)

X_smote, y_smote = smote.fit_resample(X_train, y_train)

smote_model = LogisticRegression(max_iter=1000)

smote_model.fit(X_smote, y_smote)

# 예측값
y_pred = smote_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)


print(f'Accuracy : {accuracy:.3f}')
print(f'Precision : {precision:.3f}')
print(f'Recall : {recall:.3f}')
print(f'F1-score : {f1:.3f}')

# 예측확률
y_pred_proba = smote_model.predict_proba(X_test)[:, 1] # 사기거래 1에 대한 데이터만 추출
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f'PR-AUC : {pr_auc:.4f}')

# 출력
print('\n[ SMOTE 전 / 후 비교 ]')
print(f'SMOTE 전 : {X_train.shape}')
print(f'SMOTE 후 : {X_smote.shape}')

Accuracy : 0.973
Precision : 0.056
Recall : 0.918
F1-score : 0.105
PR-AUC : 0.7188

[ SMOTE 전 / 후 비교 ]
SMOTE 전 : (227845, 28)
SMOTE 후 : (454902, 28)


### Q3. 리샘플링을 Test 데이터에도 적용하면 안 되는 이유는 무엇인가요?

**답변:**  
여기에 작성하세요.

---

# 과제 1. 불균형 처리 전후 성능과 threshold 비교

## 배경

필수 실습에서는 기본 모델과 세 가지 불균형 처리 방법을 확인했습니다.

이번 과제에서는 **기본 Logistic Regression과 class_weight 모델**을 비교하고, class_weight 모델에서 threshold에 따른 Precision / Recall 변화를 확인합니다.

> 과제는 필수 실습에서 사용한 지표와 방법을 그대로 비교하는 수준입니다.

## 요구사항

1. 기본 Logistic Regression과 `class_weight="balanced"` 모델의 다음 지표를 하나의 DataFrame으로 정리합니다.
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - PR-AUC
2. 두 모델의 Recall과 Precision 차이를 설명합니다.
3. `class_weight="balanced"` 모델의 양성 확률을 사용합니다.
4. threshold를 `0.3`, `0.5`, `0.7`로 바꾸어 예측합니다.
5. 각 threshold에서 Precision과 Recall을 계산합니다.
6. threshold 변화에 따라 두 지표가 어떻게 달라지는지 설명합니다.
7. 사기 거래를 놓치는 FN의 비용이 크다고 가정할 때, threshold를 낮추는 방향이 어떤 의미인지 설명합니다.

8. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만 사용할 때의 문제를 설명합니다.
9. FN의 비용과 데이터 특성을 고려해 `class_weight`, Random Undersampling, SMOTE 중 우선 적용할 방법을 하나 선택하고 이유와 한계를 설명합니다. 추가 모델 구현은 요구하지 않습니다.
10. Q4~Q7에 모두 답합니다.

> 처리 기법 선택은 근거 있는 제안으로 작성합니다. Test 결과를 보고 처리 기법이나 threshold를 반복 조정하지 않으며, 실제 설정 선택에는 Train 내부 검증 또는 교차검증이 필요합니다.

### 확인 포인트

- 불균형 처리 전후를 Accuracy 하나가 아니라 여러 지표로 비교했는가?
- threshold를 낮추면 Recall이 높아지는 경향을 설명할 수 있는가?
- threshold를 높이면 Precision이 높아지는 경향을 설명할 수 있는가?
- 단일한 정답 threshold가 있는 것이 아니라 비용 구조에 따라 결정해야 한다는 점을 이해했는가?
- 클래스 비율 때문에 Accuracy가 사기 탐지 성능을 오해하게 할 수 있음을 설명했는가?
- FN 비용과 데이터 특성을 근거로 처리 기법의 선택 이유·한계를 제시했는가?


In [19]:
# TODO
# 기본 모델 / class_weight 모델 비교와 threshold 0.3, 0.5, 0.7 비교를 진행하세요.

# 1. 모델 비교

models = {
    "Baseline" : LogisticRegression(random_state=42),
    "class_weight='balanced'" : LogisticRegression(class_weight="balanced", random_state=42)
} 

# 결과 저장
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "PR-AUC": average_precision_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(results)

display(comparison_df.round(4))

# 2. threshold 변화
# Balanced 모델의 양성 예측 확률
balanced_prob = models["class_weight='balanced'"].predict_proba(X_test)[:, 1]

# Threshold 설정
thresholds = [0.3, 0.5, 0.7]

threshold_results = []

for threshold in thresholds:

    y_pred = (balanced_prob >= threshold).astype(int)

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "Precision": precision,
        "Recall": recall
    })

threshold_df = pd.DataFrame(threshold_results)


print("\n[threshold 변화]")
display(threshold_df.round(4))



,Model,Accuracy,Precision,Recall,F1-score,PR-AUC
0,Baseline,0.9992,0.8289,0.6429,0.7241,0.7436
1,class_weight='balanced',0.9741,0.0579,0.9184,0.1089,0.7092



[threshold 변화]


,threshold,Precision,Recall
0,0.3,0.0259,0.9184
1,0.5,0.0579,0.9184
2,0.7,0.1146,0.8980


### Q4. 사기 거래를 놓치는 비용이 매우 크다면 threshold를 낮추는 방향은 어떤 장점과 단점이 있나요?

**답변:**  
- Threshold를 낮추면 사기 거래를 더 많이 탐지하여 FN을 줄이고 Recall을 높일 수 있지만, 정상 거래까지 사기로 판단하는 FP가 증가할 수 있다.

- 본 결과에서 threshold를 0.5에서 0.3으로 낮췄을 때 Recall은 0.9184로 유지되었지만, Precision은 0.0579에서 0.0259로 감소했다.

### Q5. CTGAN / TVAE와 SMOTE의 가장 큰 차이를 한 문장으로 설명하세요.

**답변:**  
- CTGAN과 TVAE는 모델을 학습하여 새로운 합성 데이터를 생성하지만,

- SMOTE는 가까운 소수 클래스 샘플 사이에서 새로운 데이터를 생성한다.

### Q6. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만으로 사기 탐지 성능을 판단할 때의 문제를 설명하세요.

**답변:**
- 정상 거래가 대부분인 불균형 데이터에서는 정상거래로 대부분 예측하기 때문에 Accuracy가 높게 나타날 수 있다.

- 하지만 실제 사기인 거래를 놓칠 수 있기 때문에 Recall, Precision, PR-AUC 등의 지표를 함께 보아야한다.


### Q7. 사기 거래를 놓치는 FN의 비용과 데이터 특성을 고려해 class_weight·Random Undersampling·SMOTE 중 우선 적용할 방법을 하나 선택하고, 선택 이유와 한계를 설명하세요.

**답변:**

- 사기 거래를 놓치는 FN의 비용이 크므로, 

- 기존 데이터를 유지하면서 소수 클래스에 더 높은 가중치를 부여하는 `class_weight="balanced"`를 우선 적용해본다.

- 실제 Recall은 0.6429에서 0.9184로 증가하여 사기 탐지율이 개선되었지만, 

- Precision은 0.8289에서 0.0579로 감소해 정상 거래를 사기로 잘못 판단하는 FP가 증가할 수 있다는 한계가 있다.


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 불균형 데이터에서 Accuracy 하나만 보면 위험한 이유는 무엇인가요?
2. Precision은 어떤 오류가 중요할 때 보는 지표인가요?
3. Recall은 어떤 오류가 중요할 때 보는 지표인가요?
4. class_weight, Undersampling, SMOTE는 각각 어떤 방식으로 불균형을 다루나요?
5. 리샘플링은 Train / Test 중 어디에만 적용해야 하나요?
6. threshold를 낮추면 일반적으로 Recall과 Precision은 어떻게 변하나요?